# Lightweight buffer transfer for RL-❗️QAS

Lightweight buffer transfer is a simple transfer-learning method for reinforcement learning (RL), introduced for quantum circuit optimization by Kundu and Feld (2026). Rather than starting a new task from scratch, the agent reuses a **small, useful subset** of past experience.

> Kundu, A., and S. Feld. “Replay-buffer engineering for noise-robust quantum circuit optimization.” *arXiv preprint* arXiv:2604.21863 (2026).

In RL-based quantum architecture search (RL-QAS), each interaction can be represented as

$\tau = (s, a, r, s', d)$,

where $s$ is the current circuit, $a$ is a selected gate operation, $r$ is the reward, $s'$ is the updated circuit, and $d$ indicates whether the search episode has ended.

A source task produces a replay buffer $\mathcal{B}_{\mathrm{src}}$. Lightweight transfer keeps only a compact subset,

$\widetilde{\mathcal{B}}_{\mathrm{src}} \subset \mathcal{B}_{\mathrm{src}}, \qquad |\widetilde{\mathcal{B}}_{\mathrm{src}}| = K \ll |\mathcal{B}_{\mathrm{src}}|,$

and uses it to initialize learning on a related target task:

$\mathcal{B}_{\mathrm{tgt}}^{(0)} = \widetilde{\mathcal{B}}_{\mathrm{src}}.$

## Why use it?

Quantum circuit evaluations can be costly, especially with realistic noise models, compilation constraints, or hardware experiments. Reusing promising past circuit-building decisions can help an agent learn faster and reduce the number of new evaluations required.

For example, an agent can first search for a circuit in an ideal noiseless simulator, then transfer useful circuit fragments when adapting to depolarizing noise or a hardware-aware setting.

## What should be transferred?

The transferred buffer should contain experiences that are likely to remain helpful in the new task:

- High-quality partial circuits, such as those with high fidelity.
- Gate actions that produce substantial improvement.
- Diverse circuit structures rather than many nearly identical trajectories.
- Shallow and low-entangling-gate circuits when moving to a noisy device model.

A compact score can combine quality, learning value, diversity, and target compatibility:

$w_i = \lambda_F \widehat{F}_i + \lambda_\delta \widehat{|\delta_i|} + \lambda_D \widehat{D}_i + \lambda_C \widehat{C}_i.$

Here, $F_i$ measures circuit quality, $|\delta_i|$ captures how informative a transition is for learning, $D_i$ rewards diversity, and $C_i$ measures whether the transition is likely to work in the target setting.

## Adapt gradually

Transferred experiences should help only at the beginning. As the agent collects new target-task data, it should rely increasingly on those new experiences.

One simple strategy samples a decreasing fraction of transferred transitions during training:

$\rho_u = \rho_0 \exp(-u/\tau),$

where $\rho_u$ is the source-data fraction at update $u$. This reduces the risk of **negative transfer**, where strategies learned in an ideal source setting remain overused even though they perform poorly under realistic noise.

## Key message

Lightweight buffer transfer provides a practical warm start for RL-QAS: retain a small, diverse, target-compatible memory of useful source trajectories, then let fresh target-task experience take over. It can improve early sample efficiency while keeping memory use and transfer overhead low.

<p align="center">
  <img src="buffer_transfer.png" alt="title">
</p>

In [1]:
import copy
import torch
import random
import numpy as np
import torch.nn as nn
from itertools import product
from qiskit import QuantumCircuit
from collections import namedtuple
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit_aer.noise import NoiseModel, depolarizing_error, phase_damping_error

In [2]:
def dictionary_of_actions(num_qubits):
    """
    Creates dictionary of actions for system which steers positions of gates,
    and axes of rotations.
    """
    dictionary = dict()
    i = 0
         
    for c, x in product(range(num_qubits), range(num_qubits)):
        if c != x:
            dictionary[i] = [c, x, num_qubits, 0]
            i += 1
   
    """h  denotes rotation axis. 0, 1, 2 -->  X, Y, Z axes """
    for r, h in product(range(num_qubits),
                           range(0, 4)):
        dictionary[i] = [num_qubits, 0, r, h]
        i += 1
    return dictionary

In [3]:
class RLQAS_Env:
    """
    RL-QAS environment for:
      - Bell state, 2 qubits: (|00> + |11>) / sqrt(2)
      - GHZ state,  3 qubits: (|000> + |111>) / sqrt(2)

    Noise models:
      "none"           : Ideal statevector simulation.
      "depolarizing"   : Depolarizing noise after each gate.
      "dephasing"      : Phase damping after each gate.
      "combined"       : Depolarizing followed by phase damping after each gate.

    Parameters:
      p_dep_1q   : Depolarizing strength after X, Y, Z, H.
      p_dep_2q   : Depolarizing strength after CX.
      p_deph_1q  : Phase-damping strength after X, Y, Z, H.
      p_deph_2q  : Phase-damping strength per qubit after CX.
    """

    def __init__(
        self,
        num_qubits=2,
        num_layers=4,
        fidelity_reward_scale=50.0,
        done_threshold=0.99,
        noise_model="none",
        p_dep_1q=0.001,
        p_dep_2q=0.01,
        p_deph_1q=0.001,
        p_deph_2q=0.01,
        device="cpu",
    ):

        if num_qubits not in [2, 3]:
            raise ValueError(
                "This example supports only 2-qubit Bell and 3-qubit GHZ targets."
            )

        valid_noise_models = {
            "none",
            "depolarizing",
            "dephasing",
            "combined",
        }

        if noise_model not in valid_noise_models:
            raise ValueError(
                f"noise_model must be one of {valid_noise_models}; "
                f"received '{noise_model}'."
            )

        self.num_qubits = num_qubits
        self.num_layers = num_layers
        self.device = torch.device(device)

        self.fidelity_reward_scale = fidelity_reward_scale
        self.done_threshold = done_threshold

        self.noise_model_name = noise_model
        self.p_dep_1q = p_dep_1q
        self.p_dep_2q = p_dep_2q
        self.p_deph_1q = p_deph_1q
        self.p_deph_2q = p_deph_2q

        self.state_size = (
            self.num_layers
            * self.num_qubits
            * (self.num_qubits + 4)
        )

        self.action_size = self.num_qubits * (self.num_qubits + 4)

        if self.num_qubits == 2:
            target_vector = np.array(
                [1, 0, 0, 1],
                dtype=complex,
            ) / np.sqrt(2)

        else:
            target_vector = np.array(
                [1, 0, 0, 0, 0, 0, 0, 1],
                dtype=complex,
            ) / np.sqrt(2)

        self.target_statevector = Statevector(target_vector)

        self.target_density_matrix = DensityMatrix(
            np.outer(target_vector, target_vector.conj())
        )

        self.aer_noise_model = self._build_noise_model()

        # Ideal case: use the statevector simulator.
        if self.noise_model_name == "none":
            self.sim = AerSimulator(method="statevector")

        # Noisy cases: use a density matrix because noise yields mixed states.
        else:
            self.sim = AerSimulator(
                method="density_matrix",
                noise_model=self.aer_noise_model,
            )

        self.state = None
        self.moments = None
        self.step_counter = None
        self.error = None
        self.prev_cost = None

    def _build_noise_model(self):
        """Create the requested gate-noise model."""

        if self.noise_model_name == "none":
            return None

        noise = NoiseModel()

        depol_1q = depolarizing_error(self.p_dep_1q, 1)
        depol_2q = depolarizing_error(self.p_dep_2q, 2)

        dephase_1q = phase_damping_error(self.p_deph_1q)

        # Independent phase damping on the two qubits involved in CX.
        dephase_2q = dephase_1q.tensor(dephase_1q)

        one_qubit_gates = ["x", "y", "z", "h"]
        two_qubit_gates = ["cx"]

        if self.noise_model_name == "depolarizing":
            noise.add_all_qubit_quantum_error(
                depol_1q,
                one_qubit_gates,
            )

            noise.add_all_qubit_quantum_error(
                depol_2q,
                two_qubit_gates,
            )

        elif self.noise_model_name == "dephasing":
            noise.add_all_qubit_quantum_error(
                dephase_1q,
                one_qubit_gates,
            )

            noise.add_all_qubit_quantum_error(
                dephase_2q,
                two_qubit_gates,
            )

        elif self.noise_model_name == "combined":
            combined_1q = depol_1q.compose(dephase_1q)
            combined_2q = depol_2q.compose(dephase_2q)

            noise.add_all_qubit_quantum_error(
                combined_1q,
                one_qubit_gates,
            )

            noise.add_all_qubit_quantum_error(
                combined_2q,
                two_qubit_gates,
            )

        return noise

    def reset(self):
        """Reset circuit state and return the flattened RL representation."""

        self.state = torch.zeros(
            (
                self.num_layers,
                self.num_qubits + 4,
                self.num_qubits,
            ),
            dtype=torch.float32,
        )

        self.moments = [0] * self.num_qubits
        self.step_counter = -1

        self.prev_cost = self._compute_fidelity()

        # Kept to match your original convention:
        # self.error actually stores fidelity.
        self.error = float(self.prev_cost)

        return self.state.reshape(-1).to(self.device)

    def step(self, action, train_flag=True):
        """
        action = [ctrl, targ, rot_qubit, rot_axis]

        rot_axis:
          0 -> X
          1 -> Y
          2 -> Z
          3 -> H
        """

        if isinstance(action, torch.Tensor):
            action = action.detach().cpu().tolist()

        ctrl = int(action[0])
        targ = int(action[1])
        rot_qubit = int(action[2])
        rot_axis = int(action[3])

        next_state = self.state.clone()
        self.step_counter += 1

        valid_cnot = (
            ctrl < self.num_qubits
            and targ < self.num_qubits
            and ctrl != targ
        )

        valid_oneq = (
            rot_qubit < self.num_qubits
            and 0 <= rot_axis < 4
        )

        if valid_oneq:
            gate_layer = self.moments[rot_qubit]

        elif valid_cnot:
            gate_layer = max(
                self.moments[ctrl],
                self.moments[targ],
            )

        else:
            gate_layer = 0

        if gate_layer >= self.num_layers:
            fidelity = self._compute_fidelity()

            self.prev_cost = fidelity
            self.error = float(fidelity)

            return (
                self.state.reshape(-1).to(self.device),
                torch.tensor(
                    0.0,
                    dtype=torch.float32,
                    device=self.device,
                ),
                True,
            )

        if valid_cnot:
            next_state[gate_layer, targ, ctrl] = 1.0

        if valid_oneq:
            oneq_row = self.num_qubits + rot_axis
            next_state[gate_layer, oneq_row, rot_qubit] = 1.0

        if valid_oneq:
            self.moments[rot_qubit] += 1

        elif valid_cnot:
            next_moment = max(
                self.moments[ctrl],
                self.moments[targ],
            ) + 1

            self.moments[ctrl] = next_moment
            self.moments[targ] = next_moment

        self.state = next_state

        fidelity = self._compute_fidelity()

        # Kept identical in meaning to your original code:
        # error stores fidelity.
        self.error = float(fidelity)

        reward = self._reward_from_fidelity(fidelity)
        self.prev_cost = fidelity

        done_success = fidelity >= self.done_threshold
        done_layers = self.step_counter >= self.num_layers - 1

        done = bool(done_success or done_layers)

        return (
            self.state.reshape(-1).to(self.device),
            torch.tensor(
                reward,
                dtype=torch.float32,
                device=self.device,
            ),
            done,
        )

    def _make_circuit(self):
        """Convert the RL tensor encoding into a QuantumCircuit."""

        circ = QuantumCircuit(self.num_qubits)

        state_np = self.state.detach().cpu().numpy()

        for layer in range(self.num_layers):

            cnot_block = state_np[
                layer,
                :self.num_qubits,
                :,
            ]

            targ_indices, ctrl_indices = np.where(cnot_block == 1.0)

            for targ, ctrl in zip(targ_indices, ctrl_indices):
                if ctrl != targ:
                    circ.cx(int(ctrl), int(targ))

            oneq_block = state_np[
                layer,
                self.num_qubits:self.num_qubits + 4,
                :,
            ]

            gate_rows, qubits = np.where(oneq_block == 1.0)

            for gate_row, qubit in zip(gate_rows, qubits):

                if gate_row == 0:
                    circ.x(int(qubit))

                elif gate_row == 1:
                    circ.y(int(qubit))

                elif gate_row == 2:
                    circ.z(int(qubit))

                elif gate_row == 3:
                    circ.h(int(qubit))

        return circ

    def _compute_fidelity(self):
        """
        Evaluate final target fidelity.

        Ideal case:
          statevector simulator + save_statevector.

        Noisy case:
          density-matrix simulator + save_density_matrix.
        """

        circ = self._make_circuit()

        if self.noise_model_name == "none":
            circ.save_statevector(label="final_state")

            result = self.sim.run(circ).result()

            evolved_state = Statevector(
                result.data(0)["final_state"]
            )

            fidelity = state_fidelity(
                evolved_state,
                self.target_statevector,
            )

        else:
            circ.save_density_matrix(label="final_state")

            result = self.sim.run(circ).result()

            evolved_state = DensityMatrix(
                result.data(0)["final_state"]
            )

            fidelity = state_fidelity(
                evolved_state,
                self.target_density_matrix,
            )

        return float(fidelity)

    def _reward_from_fidelity(self, x=None):
        """
        Original reward behavior retained:
        self.error stores fidelity.
        """

        scalar = 50

        if self.error >= self.done_threshold:
            rwd = scalar * self.error

        else:
            rwd = self.error

        return rwd

## Noiseless training to store experience in buffer

In [4]:
num_qubits, max_steps, num_episodes = 3, 8, 2000

"""
IDEAL
"""
env_noiseless = RLQAS_Env(num_qubits=num_qubits, num_layers=max_steps, noise_model="none",)

"""
Depolarizing
"""
env_depolarizing = RLQAS_Env(
    num_qubits=num_qubits,
    num_layers=max_steps,
    noise_model="depolarizing",
    p_dep_1q=0.001,
    p_dep_2q=0.01,
)

"""
Dephasing
"""
env_dephasing = RLQAS_Env(
    num_qubits=num_qubits,
    num_layers=max_steps,
    noise_model="dephasing",
    p_deph_1q=0.002,
    p_deph_2q=0.01,
)

"""
Deplorizing & dephasing
"""
env_noise_combined = RLQAS_Env(
    num_qubits=num_qubits,
    num_layers=max_steps,
    noise_model="combined",
    p_dep_1q=0.001,
    p_dep_2q=0.01,
    p_deph_1q=0.002,
    p_deph_2q=0.001,
    done_threshold=0.98,
)


In [5]:
import os
SEED = 42

def seed_everything(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)

    # Python RNG: epsilon-greedy exploration + replay-buffer sampling
    random.seed(seed)

    # NumPy RNG: important if env/reset/step uses np.random
    np.random.seed(seed)

    # PyTorch RNG: DQN initialization and torch random operations
    torch.manual_seed(seed)

    # Safe to keep if you later move to CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Optional: stricter deterministic behavior
    torch.use_deterministic_algorithms(True)

    # Relevant for CUDA/cuDNN, harmless to include for a CPU-only run
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

In [10]:
from pathlib import Path
import random

import torch
import torch.nn as nn
import torch.optim as optim

num_qubits, max_steps, num_episodes = 3, 8, 2000

actions_dict = dictionary_of_actions(num_qubits)
num_actions = len(actions_dict)
device = torch.device("cpu")


environment_type_list = ["none", "deplorizing", "dephasing", "combined"]

environment_type = environment_type_list[0]

print('--------------------------')
if environment_type == environment_type_list[0]:
    env = env_noiseless
    comment = "😇 Noiseless Mode Activated 😇"
elif environment_type == environment_type_list[1]:
    env = env_depolarizing
    comment = "😬 Depolarizing Noise Activated! Need to work harder! 😬"
elif environment_type == environment_type_list[2]:
    env = env_dephasing
    comment = "😬 Dephasing Noise Activated! Need to work harder! 😬"
elif environment_type == environment_type_list[3]:
    env = env_noise_combined
    comment = "😤 Both Noise Models Are Activated! Need to work much harder! 😤"
else:
    raise ValueError(f"Unknown environment_type: {environment_type}")

print(comment)
print("SEARCHING WITH JUST ONE LAYER NEURAL NETWORK UNDER RL-FRAMEWORK!")
print('--------------------------')
print()

state_dim = env.state_size
action_dim = num_actions


class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)


q_net = DQN(state_dim, action_dim).to(device)
target_net = DQN(state_dim, action_dim).to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

# --- replay buffer ---
replay_buffer = []
buffer_capacity = 10000
batch_size = 64
gamma = 0.99

# Save a buffer snapshot after every 500 completed episodes.
save_every_episodes = 500
replay_buffer_dir = Path("saved_replay_buffers")
replay_buffer_dir.mkdir(parents=True, exist_ok=True)


def push_transition(transition):
    if len(replay_buffer) >= buffer_capacity:
        replay_buffer.pop(0)
    replay_buffer.append(transition)


def sample_batch():
    batch = random.sample(replay_buffer, batch_size)
    state_b, action_b, reward_b, next_state_b, done_b = zip(*batch)

    state_b = torch.stack(state_b)
    action_b = torch.tensor(action_b, dtype=torch.long, device=device)
    reward_b = torch.tensor(reward_b, dtype=torch.float32, device=device)
    next_state_b = torch.stack(next_state_b)
    done_b = torch.tensor(done_b, dtype=torch.float32, device=device)

    return state_b, action_b, reward_b, next_state_b, done_b


def save_replay_buffer(episode):
    """Save a CPU-only snapshot of the replay buffer and run metadata."""
    replay_buffer_cpu = [
        (
            state.detach().cpu().clone(),
            int(action_idx),
            float(reward),
            next_state.detach().cpu().clone(),
            float(done),
        )
        for state, action_idx, reward, next_state, done in replay_buffer
    ]

    checkpoint = {
        "episode": int(episode),
        "global_step": int(global_step),
        "num_qubits": int(num_qubits),
        "environment_type": str(environment_type),
        "state_dim": int(state_dim),
        "action_dim": int(action_dim),
        "buffer_capacity": int(buffer_capacity),
        "replay_buffer_size": len(replay_buffer_cpu),
        "replay_buffer": replay_buffer_cpu,
    }

    save_path = replay_buffer_dir / (
        f"replay_buffer_nq{num_qubits}_{environment_type}_ep{episode:05d}.pt"
    )
    torch.save(checkpoint, save_path)

    print(
        f"Saved replay buffer: {save_path} | "
        f"transitions = {len(replay_buffer_cpu)}"
    )


# --- epsilon-greedy parameters ---
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay = 10000
global_step = 0


def get_epsilon(step):
    return max(epsilon_end, epsilon_start - step / epsilon_decay)


def select_action(state_vec):
    global global_step

    epsilon = get_epsilon(global_step)
    global_step += 1

    if random.random() < epsilon:
        return random.randint(0, action_dim - 1)

    with torch.no_grad():
        q_values = q_net(state_vec.unsqueeze(0))
        return int(q_values.argmax(dim=1).item())


# --- training loop ---
max_steps_per_episode = max_steps
target_update_freq = 100

episode_rewards = []
episode_final_fidelities = []
episode_final_epsilons = []
episode_num_gates = []

for ep in range(num_episodes):
    state = env.reset().to(device)
    total_reward = 0.0
    total_gates = 0

    for t in range(max_steps_per_episode):
        a_idx = select_action(state)
        action = actions_dict[a_idx]

        next_state, reward_t, done = env.step(action)
        next_state = next_state.to(device)

        r = reward_t.item()
        total_reward += r

        total_gates = int((env.state.detach().cpu().numpy() != 0).sum())

        # Store detached CPU tensors so the replay buffer never retains
        # autograd graphs and remains portable across devices.
        push_transition((
            state.detach().cpu().clone(),
            a_idx,
            r,
            next_state.detach().cpu().clone(),
            float(done),
        ))

        state = next_state

        if len(replay_buffer) >= batch_size:
            s_b, a_b, r_b, ns_b, d_b = sample_batch()
            s_b = s_b.to(device)
            ns_b = ns_b.to(device)

            q_values = q_net(s_b)
            q_sa = q_values.gather(1, a_b.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                next_q_values = target_net(ns_b)
                max_next_q = next_q_values.max(dim=1)[0]
                target_q = r_b + gamma * max_next_q * (1.0 - d_b)

            loss = loss_fn(q_sa, target_q)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    episode_rewards.append(total_reward)
    episode_final_fidelities.append(env.prev_cost)
    episode_num_gates.append(total_gates)
    episode_final_epsilons.append(get_epsilon(global_step))

    if (ep + 1) % target_update_freq == 0:
        target_net.load_state_dict(q_net.state_dict())

    if (ep + 1) % save_every_episodes == 0:
        save_replay_buffer(episode=ep + 1)

    if (ep + 1) % 10 == 0:
        print(
            f"Episode {ep + 1:4d} | total reward = {total_reward:.3f} | "
            f"last fidelity = {env.prev_cost:.6f} | gates = {total_gates} | "
            f"epsilon = {episode_final_epsilons[-1]:.3f}"
        )

print("Training finished.")

# --- quick greedy evaluation run ---
state = env.reset().to(device)
for t in range(max_steps_per_episode):
    with torch.no_grad():
        q_vals = q_net(state.unsqueeze(0))
        a_idx = int(q_vals.argmax(dim=1).item())

    action = actions_dict[a_idx]
    next_state, reward_t, done = env.step(action)
    state = next_state.to(device)

    print(
        f"Eval step {t + 1}: action_id={a_idx}, action={action}, "
        f"fidelity={env.prev_cost:.6f}, reward={reward_t.item():.3f}, done={done}"
    )

    if done:
        break

--------------------------
😇 Noiseless Mode Activated 😇
SEARCHING WITH JUST ONE LAYER NEURAL NETWORK UNDER RL-FRAMEWORK!
--------------------------

Episode   10 | total reward = 2.250 | last fidelity = 0.000000 | gates = 8 | epsilon = 0.992
Episode   20 | total reward = 1.625 | last fidelity = 0.125000 | gates = 8 | epsilon = 0.984
Episode   30 | total reward = 3.750 | last fidelity = 0.250000 | gates = 8 | epsilon = 0.976
Episode   40 | total reward = 1.750 | last fidelity = 0.000000 | gates = 8 | epsilon = 0.968
Episode   50 | total reward = 0.000 | last fidelity = 0.000000 | gates = 8 | epsilon = 0.960
Episode   60 | total reward = 1.250 | last fidelity = 0.000000 | gates = 8 | epsilon = 0.952
Episode   70 | total reward = 1.500 | last fidelity = 0.500000 | gates = 8 | epsilon = 0.944
Episode   80 | total reward = 1.250 | last fidelity = 0.250000 | gates = 8 | epsilon = 0.936
Episode   90 | total reward = 1.500 | last fidelity = 0.500000 | gates = 8 | epsilon = 0.928
Episode  100 |

## After buffer transfer

In [11]:
num_qubits, max_steps, num_episodes = 3, 8, 5000

actions_dict = dictionary_of_actions(num_qubits)
num_actions = len(actions_dict)
device = torch.device("cpu")

environment_type_list = ["none", "deplorizing", "dephasing", "combined"]

# ------------------------------------------------------------
# Choose the target environment.
#
# Example:
# environment_type = environment_type_list[1]
# means train in the depolarizing-noise environment.
# ------------------------------------------------------------
environment_type = environment_type_list[3]

# ------------------------------------------------------------
# Replay-buffer warm-start configuration.
#
# Set to None for ordinary training with an initially empty buffer.
#
# For transfer from a 3-qubit noiseless source run:
warm_start_buffer_path = Path(
    "saved_replay_buffers/replay_buffer_nq3_none_ep02000.pt"
)
# ------------------------------------------------------------
# warm_start_buffer_path = None

# Maximum number of loaded source transitions to retain.
#
# None: load all stored source transitions, limited only by
#       buffer_capacity.
max_transferred_transitions = 1000

# If True and the source buffer is larger than
# max_transferred_transitions, draw a random subset.
# If False, retain the most recently stored source transitions.
randomly_subsample_transferred_buffer = True


# ============================================================
# Select environment
# ============================================================
print("--------------------------")

if environment_type == environment_type_list[0]:
    env = env_noiseless
    comment = "😇 Noiseless Mode Activated 😇"

elif environment_type == environment_type_list[1]:
    env = env_depolarizing
    comment = "😬 Depolarizing Noise Activated! Need to work harder! 😬"

elif environment_type == environment_type_list[2]:
    env = env_dephasing
    comment = "😬 Dephasing Noise Activated! Need to work harder! 😬"

elif environment_type == environment_type_list[3]:
    env = env_noise_combined
    comment = "😤 Both Noise Models Are Activated! Need to work much harder! 😤"

else:
    raise ValueError(f"Unknown environment_type: {environment_type}")

print(comment)
print("SEARCHING WITH JUST ONE LAYER NEURAL NETWORK UNDER RL-FRAMEWORK!")
print("--------------------------")
print()

state_dim = env.state_size
action_dim = num_actions


# ============================================================
# DQN model
# ============================================================
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)


q_net = DQN(state_dim, action_dim).to(device)
target_net = DQN(state_dim, action_dim).to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()


# ============================================================
# Replay buffer and checkpoint settings
# ============================================================
replay_buffer = []
buffer_capacity = 10000
batch_size = 64
gamma = 0.99

save_every_episodes = 500
replay_buffer_dir = Path("saved_replay_buffers")
replay_buffer_dir.mkdir(parents=True, exist_ok=True)


def make_cpu_transition(state, action_idx, reward, next_state, done):
    """
    Convert a transition to a detached, portable CPU representation.

    The saved and replayed states have no computation graph attached.
    """
    return (
        state.detach().cpu().clone(),
        int(action_idx),
        float(reward),
        next_state.detach().cpu().clone(),
        float(done),
    )


def push_transition(transition):
    """Append one transition and remove the oldest if capacity is full."""
    if len(replay_buffer) >= buffer_capacity:
        replay_buffer.pop(0)

    replay_buffer.append(transition)


def sample_batch():
    """Sample uniformly from source and target transitions in one buffer."""
    batch = random.sample(replay_buffer, batch_size)

    state_b, action_b, reward_b, next_state_b, done_b = zip(*batch)

    state_b = torch.stack(state_b).to(device)
    action_b = torch.tensor(action_b, dtype=torch.long, device=device)
    reward_b = torch.tensor(reward_b, dtype=torch.float32, device=device)
    next_state_b = torch.stack(next_state_b).to(device)
    done_b = torch.tensor(done_b, dtype=torch.float32, device=device)

    return state_b, action_b, reward_b, next_state_b, done_b


def load_warm_start_buffer(checkpoint_path):
    """
    Load and validate an existing replay-buffer checkpoint.

    Required checkpoint keys:
    - num_qubits
    - state_dim
    - action_dim
    - replay_buffer
    """

    if checkpoint_path is None:
        print("Replay-buffer warm start disabled: empty buffer.")
        return []

    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Could not find warm-start replay buffer:\n{checkpoint_path}"
        )

    # Only load checkpoint files that you generated or trust.
    # map_location='cpu' allows loading regardless of original device.
    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )
    except TypeError:
        # Allows compatibility with older PyTorch versions.
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu"
        )

    required_keys = {
        "num_qubits",
        "state_dim",
        "action_dim",
        "replay_buffer",
    }

    missing_keys = required_keys.difference(checkpoint.keys())

    if missing_keys:
        raise KeyError(
            "Replay-buffer checkpoint is missing fields: "
            f"{sorted(missing_keys)}"
        )

    saved_num_qubits = int(checkpoint["num_qubits"])
    saved_state_dim = int(checkpoint["state_dim"])
    saved_action_dim = int(checkpoint["action_dim"])

    if saved_num_qubits != num_qubits:
        raise ValueError(
            "Incompatible replay buffer: num_qubits mismatch.\n"
            f"Checkpoint: {saved_num_qubits}\n"
            f"Current:    {num_qubits}"
        )

    if saved_state_dim != state_dim:
        raise ValueError(
            "Incompatible replay buffer: state_dim mismatch.\n"
            f"Checkpoint: {saved_state_dim}\n"
            f"Current:    {state_dim}"
        )

    if saved_action_dim != action_dim:
        raise ValueError(
            "Incompatible replay buffer: action_dim mismatch.\n"
            f"Checkpoint: {saved_action_dim}\n"
            f"Current:    {action_dim}"
        )

    loaded_buffer = checkpoint["replay_buffer"]

    if not loaded_buffer:
        print("Warm-start checkpoint contains no transitions.")
        return []

    # Validate every transition and copy all tensors to CPU.
    validated_buffer = []

    for transition in loaded_buffer:
        if len(transition) != 5:
            raise ValueError(
                "A loaded transition must be "
                "(state, action_idx, reward, next_state, done)."
            )

        state, action_idx, reward, next_state, done = transition

        state = torch.as_tensor(
            state,
            dtype=torch.float32
        ).detach().cpu().clone().reshape(-1)

        next_state = torch.as_tensor(
            next_state,
            dtype=torch.float32
        ).detach().cpu().clone().reshape(-1)

        if state.numel() != state_dim:
            raise ValueError(
                f"Loaded state has {state.numel()} values; "
                f"expected state_dim={state_dim}."
            )

        if next_state.numel() != state_dim:
            raise ValueError(
                f"Loaded next_state has {next_state.numel()} values; "
                f"expected state_dim={state_dim}."
            )

        action_idx = int(action_idx)

        if not 0 <= action_idx < action_dim:
            raise ValueError(
                f"Loaded action {action_idx} is invalid for "
                f"action_dim={action_dim}."
            )

        validated_buffer.append((
            state,
            action_idx,
            float(reward),
            next_state,
            float(done),
        ))

    # Select the requested number of source transitions.
    if max_transferred_transitions is None:
        transfer_limit = buffer_capacity
    else:
        transfer_limit = min(
            int(max_transferred_transitions),
            buffer_capacity
        )

    if transfer_limit <= 0:
        print("Transfer limit is zero: empty warm-start buffer.")
        return []

    if len(validated_buffer) > transfer_limit:
        if randomly_subsample_transferred_buffer:
            validated_buffer = random.sample(
                validated_buffer,
                transfer_limit
            )
        else:
            validated_buffer = validated_buffer[-transfer_limit:]

    print("Replay buffer warm start completed:")
    print(f"  Source checkpoint:   {checkpoint_path}")
    print(
        "  Source environment:  "
        f"{checkpoint.get('environment_type', 'unknown')}"
    )
    print(
        "  Source episode:      "
        f"{checkpoint.get('episode', 'unknown')}"
    )
    print(f"  Qubits:              {saved_num_qubits}")
    print(f"  Loaded transitions:  {len(validated_buffer)}")
    print()

    return validated_buffer


def save_replay_buffer(episode):
    """Save the current replay buffer and metadata for later transfer."""

    replay_buffer_cpu = [
        make_cpu_transition(
            state,
            action_idx,
            reward,
            next_state,
            done,
        )
        for state, action_idx, reward, next_state, done in replay_buffer
    ]

    checkpoint = {
        "episode": int(episode),
        "global_step": int(global_step),
        "num_qubits": int(num_qubits),
        "environment_type": str(environment_type),
        "state_dim": int(state_dim),
        "action_dim": int(action_dim),
        "buffer_capacity": int(buffer_capacity),
        "replay_buffer_size": len(replay_buffer_cpu),
        "replay_buffer": replay_buffer_cpu,
    }

    save_path = replay_buffer_dir / (
        f"replay_buffer_nq{num_qubits}_{environment_type}_ep{episode:05d}.pt"
    )

    torch.save(checkpoint, save_path)

    print(
        f"Saved replay buffer: {save_path} | "
        f"transitions = {len(replay_buffer_cpu)}"
    )


# ============================================================
# Initialize target replay buffer from source checkpoint
# ============================================================
replay_buffer = load_warm_start_buffer(warm_start_buffer_path)

# Safety limit: this is normally already enforced during loading.
if len(replay_buffer) > buffer_capacity:
    replay_buffer = replay_buffer[-buffer_capacity:]


# ============================================================
# Epsilon-greedy policy
# ============================================================
epsilon_start = 0.3
epsilon_end = 0.05
epsilon_decay = 2000
global_step = 0


def get_epsilon(step):
    return max(epsilon_end, epsilon_start - step / epsilon_decay)


def select_action(state_vec):
    global global_step

    epsilon = get_epsilon(global_step)
    global_step += 1

    if random.random() < epsilon:
        return random.randint(0, action_dim - 1)

    with torch.no_grad():
        q_values = q_net(state_vec.unsqueeze(0))
        return int(q_values.argmax(dim=1).item())


# ============================================================
# Training loop
# ============================================================
max_steps_per_episode = max_steps
target_update_freq = 100

episode_rewards_w_t = []
episode_final_fidelities_w_t = []
episode_final_epsilons_w_t = []
episode_num_gates_w_t = []

for ep in range(num_episodes):
    state = env.reset().to(device)

    total_reward = 0.0
    total_gates = 0

    for t in range(max_steps_per_episode):
        a_idx = select_action(state)
        action = actions_dict[a_idx]

        next_state, reward_t, done = env.step(action)
        next_state = next_state.to(device)

        r = reward_t.item()
        total_reward += r

        total_gates = int(
            (env.state.detach().cpu().numpy() != 0).sum()
        )

        # Add target-task experience to the replay buffer.
        # When the buffer reaches capacity, the oldest transition is removed.
        push_transition(
            make_cpu_transition(
                state=state,
                action_idx=a_idx,
                reward=r,
                next_state=next_state,
                done=done,
            )
        )

        state = next_state

        # The batch can include both transferred source transitions
        # and newly collected target transitions.
        if len(replay_buffer) >= batch_size:
            s_b, a_b, r_b, ns_b, d_b = sample_batch()

            q_values = q_net(s_b)
            q_sa = q_values.gather(
                1,
                a_b.unsqueeze(1)
            ).squeeze(1)

            with torch.no_grad():
                next_q_values = target_net(ns_b)
                max_next_q = next_q_values.max(dim=1)[0]

                target_q = (
                    r_b
                    + gamma * max_next_q * (1.0 - d_b)
                )

            loss = loss_fn(q_sa, target_q)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    episode_rewards_w_t.append(total_reward)
    episode_final_fidelities_w_t.append(env.prev_cost)
    episode_num_gates_w_t.append(total_gates)
    episode_final_epsilons_w_t.append(get_epsilon(global_step))

    if (ep + 1) % target_update_freq == 0:
        target_net.load_state_dict(q_net.state_dict())

    # if (ep + 1) % save_every_episodes == 0:
    #     save_replay_buffer(episode=ep + 1)

    if (ep + 1) % 10 == 0:
        print(
            f"Episode {ep + 1:4d} | "
            f"total reward = {total_reward:.3f} | "
            f"last fidelity = {env.prev_cost:.6f} | "
            f"gates = {total_gates} | "
            f"epsilon = {episode_final_epsilons[-1]:.3f} | "
            f"buffer = {len(replay_buffer)}"
        )

print("Training finished.")


# ============================================================
# Greedy evaluation
# ============================================================
state = env.reset().to(device)

for t in range(max_steps_per_episode):
    with torch.no_grad():
        q_vals = q_net(state.unsqueeze(0))
        a_idx = int(q_vals.argmax(dim=1).item())

    action = actions_dict[a_idx]

    next_state, reward_t, done = env.step(action)
    state = next_state.to(device)

    print(
        f"Eval step {t + 1}: "
        f"action_id={a_idx}, action={action}, "
        f"fidelity={env.prev_cost:.6f}, "
        f"reward={reward_t.item():.3f}, done={done}"
    )

    if done:
        break

--------------------------
😤 Both Noise Models Are Activated! Need to work much harder! 😤
SEARCHING WITH JUST ONE LAYER NEURAL NETWORK UNDER RL-FRAMEWORK!
--------------------------



FileNotFoundError: Could not find warm-start replay buffer:
saved_replay_buffers/replay_buffer_nq3_none_ep02000.pt

## Without transfer

In [8]:
num_qubits, max_steps, num_episodes = 3, 8, 5000

import torch.optim as optim

num_actions = len(actions_dict)
device = torch.device("cpu")


environment_type_list = ["none", "deplorizing", "dephasing", "combined"]
environment_type = environment_type_list[3]

print('--------------------------')
if environment_type == environment_type_list[0]:
    env = env_noiseless
    comment = "😇 Noiseless Mode Activated 😇"
elif environment_type == environment_type_list[1]:
    env = env_depolarizing
    comment = "😬 Depolarizing Noise Activated! Need to work harder! 😬"
elif environment_type == environment_type_list[2]:
    env = env_dephasing
    comment = "😬 Dephasing Noise Activated! Need to work harder! 😬"
elif environment_type == environment_type_list[3]:
    env = env_noise_combined
    comment = "😤 Both Noise Models Are Activated! Need to work much harder! 😤"
print(comment)
print("SEARCHIGN WITH JUST ONE LAYER NEURAL NETWORK UNDER RL-FRAMEWORK!")
print('--------------------------')
print()

state_dim = env.state_size  # flattened state length
action_dim = num_actions    # discrete actions 0..9

# --- tiny DQN network ---
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)

q_net = DQN(state_dim, action_dim).to(device)
target_net = DQN(state_dim, action_dim).to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

# --- simple replay buffer ---
replay_buffer = []
buffer_capacity = 10000
batch_size = 64
gamma = 0.99

def push_transition(transition):
    if len(replay_buffer) >= buffer_capacity:
        replay_buffer.pop(0)
    replay_buffer.append(transition)

def sample_batch():
    batch = random.sample(replay_buffer, batch_size)
    state_b, action_b, reward_b, next_state_b, done_b = zip(*batch)
    state_b = torch.stack(state_b)
    action_b = torch.tensor(action_b, dtype=torch.long, device=device)
    reward_b = torch.tensor(reward_b, dtype=torch.float32, device=device)
    next_state_b = torch.stack(next_state_b)
    done_b = torch.tensor(done_b, dtype=torch.float32, device=device)
    return state_b, action_b, reward_b, next_state_b, done_b

# --- epsilon-greedy parameters ---
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay = 10000   # in steps
global_step = 0

def get_epsilon(step):
    return max(epsilon_end, epsilon_start - step / epsilon_decay)

def select_action(state_vec):
    global global_step
    epsilon = get_epsilon(global_step)
    global_step += 1
    if random.random() < epsilon:
        return random.randint(0, action_dim - 1)
    with torch.no_grad():
        q_values = q_net(state_vec.unsqueeze(0))
        return int(q_values.argmax(dim=1).item())

# --- training loop ---
max_steps_per_episode = max_steps
target_update_freq = 100  # episodes

episode_rewards_wo_t = []
episode_final_fidelities_wo_t = []
episode_final_epsilons_wo_t = []
episode_num_gates_wo_t = []  # total gates used in that episode

for ep in range(num_episodes):
    state = env.reset()                         # shape: [state_dim], already on device
    state = state.to(device)
    total_reward = 0.0
    total_gates = 0

    for t in range(max_steps_per_episode):
        # choose action index via epsilon-greedy
        a_idx = select_action(state)
        action = actions_dict[a_idx]

        # step environment
        next_state, reward_t, done = env.step(action)
        next_state = next_state.to(device)

        r = reward_t.item()
        total_reward += r

        # count how many gates are present in the circuit state after this step
        # (you can also use env.moments if you prefer; here we just count nonzeros)
        total_gates = int((env.state.numpy() != 0).sum())

        # store transition in replay buffer
        push_transition((
            state,
            a_idx,
            r,
            next_state,
            float(done)
        ))

        state = next_state

        # DQN update
        if len(replay_buffer) >= batch_size:
            s_b, a_b, r_b, ns_b, d_b = sample_batch()

            # current Q(s, a)
            q_values = q_net(s_b)
            q_sa = q_values.gather(1, a_b.unsqueeze(1)).squeeze(1)

            # target Q = r + gamma * max_a' Q_target(s', a') * (1 - done)
            with torch.no_grad():
                next_q_values = target_net(ns_b)
                max_next_q = next_q_values.max(dim=1)[0]
                target_q = r_b + gamma * max_next_q * (1.0 - d_b)

            loss = loss_fn(q_sa, target_q)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break #requires applicants to provide a

    # episode-level logging
    episode_rewards_wo_t.append(total_reward)
    episode_final_fidelities_wo_t.append(env.prev_cost)
    episode_num_gates_wo_t.append(total_gates)
    episode_final_epsilons_wo_t.append(get_epsilon(global_step))

    # update target network
    if (ep + 1) % target_update_freq == 0:
        target_net.load_state_dict(q_net.state_dict())

    if (ep + 1) % 10 == 0:
        print(f"Episode {ep+1:3d} | total reward = {total_reward:.3f} | "
              f"last fidelity = {env.prev_cost:.6f} | gates = {total_gates} | "
              f"epsilon = {episode_final_epsilons_wo_t[-1]:.3f}")

print("Training finished.")

# quick greedy evaluation run
state = env.reset().to(device)
for t in range(max_steps_per_episode):
    with torch.no_grad():
        q_vals = q_net(state.unsqueeze(0))
        a_idx = int(q_vals.argmax(dim=1).item())
    action = actions_dict[a_idx]
    next_state, reward_t, done = env.step(action)
    state = next_state.to(device)
    print(f"Eval step {t+1}: action_id={a_idx}, action={action}, fidelity={env.prev_cost:.6f}, reward={reward_t.item():.3f}, done={done}")
    if done:
        break

--------------------------
😤 Both Noise Models Are Activated! Need to work much harder! 😤
SEARCHIGN WITH JUST ONE LAYER NEURAL NETWORK UNDER RL-FRAMEWORK!
--------------------------

Episode  10 | total reward = 1.497 | last fidelity = 0.002977 | gates = 8 | epsilon = 0.992
Episode  20 | total reward = 0.513 | last fidelity = 0.002730 | gates = 8 | epsilon = 0.984
Episode  30 | total reward = 1.736 | last fidelity = 0.005046 | gates = 8 | epsilon = 0.976
Episode  40 | total reward = 0.502 | last fidelity = 0.000499 | gates = 8 | epsilon = 0.968
Episode  50 | total reward = 2.476 | last fidelity = 0.002855 | gates = 8 | epsilon = 0.960
Episode  60 | total reward = 1.001 | last fidelity = 0.248503 | gates = 8 | epsilon = 0.952
Episode  70 | total reward = 1.489 | last fidelity = 0.486924 | gates = 8 | epsilon = 0.944
Episode  80 | total reward = 1.986 | last fidelity = 0.005140 | gates = 8 | epsilon = 0.936
Episode  90 | total reward = 1.002 | last fidelity = 0.244834 | gates = 8 | epsil

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

chunk_size = 100
threshold = 0.98

# Use the actual number of completed episodes. This also works if a run
# ends early or you later change num_episodes.
n_wo_t = len(episode_final_fidelities_wo_t)
n_w_t = len(episode_final_fidelities_w_t)

if n_wo_t == 0:
    raise ValueError("episode_final_fidelities_wo_t is empty.")

if n_w_t == 0:
    raise ValueError("episode_final_fidelities_w_t is empty.")

if n_wo_t != n_w_t:
    raise ValueError(
        "The warm-transfer and no-transfer runs have different numbers "
        f"of logged episodes: {n_w_t} versus {n_wo_t}."
    )

num_chunks = (n_wo_t + chunk_size - 1) // chunk_size

chunk_success_rates_agent_wo_t = []
chunk_success_rates_agent_w_t = []
chunk_episode_centers = []

for k in range(num_chunks):
    start = k * chunk_size
    end = min((k + 1) * chunk_size, n_wo_t)

    fidelities_chunk_agent_wo_t = episode_final_fidelities_wo_t[start:end]
    fidelities_chunk_agent_w_t = episode_final_fidelities_w_t[start:end]

    success_rate_wo_t = np.mean(
        np.asarray(fidelities_chunk_agent_wo_t) >= threshold
    )
    success_rate_w_t = np.mean(
        np.asarray(fidelities_chunk_agent_w_t) >= threshold
    )

    chunk_success_rates_agent_wo_t.append(float(success_rate_wo_t))
    chunk_success_rates_agent_w_t.append(float(success_rate_w_t))

    chunk_episode_centers.append(end)

# print("No-transfer success rates:", chunk_success_rates_agent_wo_t)
# print("Warm-transfer success rates:", chunk_success_rates_agent_w_t)

plt.figure(figsize=(8, 4))

plt.plot(
    chunk_episode_centers,
    chunk_success_rates_agent_wo_t,
    "-o",
    markersize=6,
    linewidth=2,
    label="Without transfer"
)

plt.plot(
    chunk_episode_centers,
    chunk_success_rates_agent_w_t,
    "-x",
    markersize=7,
    linewidth=2,
    label="With lightweight buffer transfer"
)

plt.axhline(
    y=0.0,
    color="gray",
    linewidth=0.8,
    alpha=0.5
)

plt.xlabel(f"Training episode (success rate per {chunk_size} episodes)")
plt.ylabel(f"Success rate: final fidelity ≥ {threshold}")
plt.ylim(-0.02, 1.02)
plt.xlim(0, max(chunk_episode_centers) + chunk_size / 2)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
# plt.show()

In [ ]:
chunk_success_rates_agent_wo_t[-1], chunk_success_rates_agent_w_t[-1]